# FIFA World Cup 2026 Tournament Simulator - Real Data Only Colab Notebook

This notebook trains the project using real football data only. It downloads multiple real CSVs, derives rankings from the match history, validates the inputs, and stops if any required file is missing.

## What this notebook uses

- `results.csv` for historical match results
- `goalscorers.csv` for scorer-level context
- `shootouts.csv` for penalty shootout history
- `former_names.csv` for team name history
- A rankings table derived from the real match results with Elo

There is no synthetic fallback in this notebook. If the required files are missing, the notebook stops so you can fix the data source first.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import os

PROJECT_DIR = Path('/content/world-cup-2026-simulator')
DRIVE_DIR = Path('/content/drive/MyDrive/world-cup-2026-simulator')
RAW_DIR = PROJECT_DIR / 'data' / 'raw'
MODELS_DIR = PROJECT_DIR / 'models'
DRIVE_RAW_DIR = DRIVE_DIR / 'data' / 'raw'
DRIVE_MODELS_DIR = DRIVE_DIR / 'models'
RAW_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [6]:
# Clone or update the repository in Colab.
if not PROJECT_DIR.exists():
    !git clone https://github.com/musa-qureshi/world-cup-2026-simulator.git /content/world-cup-2026-simulator
else:
    %cd /content/world-cup-2026-simulator
    !git pull

%cd /content/world-cup-2026-simulator

/content/world-cup-2026-simulator
Already up to date.
/content/world-cup-2026-simulator


In [7]:
# Install project dependencies and the Kaggle client.
!pip -q install -r requirements.txt
!pip -q install kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 65.8 MB/s eta 0:00:00


## Download real datasets

This downloads the international football history dataset from Kaggle, which includes the match results plus the related `goalscorers`, `shootouts`, and `former_names` CSV files.

In [9]:
# Authenticate Kaggle using kaggle.json from either /content or Google Drive.
kaggle_token = Path('/content/kaggle.json')
if not kaggle_token.exists():
    drive_token = Path('/content/drive/MyDrive/kaggle.json')
    if drive_token.exists():
        shutil.copy2(drive_token, kaggle_token)

if not kaggle_token.exists():
    raise FileNotFoundError('kaggle.json not found. Upload it to Colab or place it in Google Drive at MyDrive/kaggle.json.')

!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d martj42/international-football-results-from-1872-to-2017 -p /content/world-cup-2026-simulator/data/raw --unzip

Dataset URL: https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017
License(s): CC0-1.0
100% 1.21M/1.21M [00:00<00:00, 86.2MB/s]



In [10]:
# Validate the downloaded real files before training.
required_files = ['results.csv', 'goalscorers.csv', 'shootouts.csv', 'former_names.csv']
missing_files = [name for name in required_files if not (RAW_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required Kaggle files: {missing_files}')

print('Downloaded files:')
for name in sorted(os.listdir(RAW_DIR)):
    print('-', name)

Downloaded files:
- former_names.csv
- goalscorers.csv
- results.csv
- shootouts.csv


In [11]:
# Prepare the files expected by the training pipeline.
from src.preprocessing import load_match_data
from src.elo_rating import EloRatingSystem
import pandas as pd

match_source = RAW_DIR / 'results.csv'
matches = load_match_data(match_source)
matches.to_csv(RAW_DIR / 'matches.csv', index=False)
matches.to_csv(RAW_DIR / 'international_results.csv', index=False)

elo = EloRatingSystem()
elo.bulk_fit(matches)
rankings = elo.to_frame().sort_values('rating', ascending=False).reset_index(drop=True)
rankings.insert(1, 'rank', range(1, len(rankings) + 1))
rankings['confederation'] = 'Unknown'
rankings = rankings[['team', 'rank', 'rating', 'confederation']]
rankings.to_csv(RAW_DIR / 'fifa_rankings.csv', index=False)
rankings.to_csv(RAW_DIR / 'rankings.csv', index=False)

print('Prepared matches.csv, international_results.csv, fifa_rankings.csv, and rankings.csv from real match data.')
print('Match rows:', len(matches))
print('Ranking rows:', len(rankings))

Prepared matches.csv, international_results.csv, fifa_rankings.csv, and rankings.csv from real match data.
Match rows: 49287
Ranking rows: 333


In [12]:
# Inspect the extra real CSVs so you know what is available for future feature work.
import pandas as pd
for filename in ['goalscorers.csv', 'shootouts.csv', 'former_names.csv']:
    frame = pd.read_csv(RAW_DIR / filename)
    print(f'\n{filename}: {frame.shape[0]} rows x {frame.shape[1]} columns')
    print(list(frame.columns))


goalscorers.csv: 47601 rows x 8 columns
['date', 'home_team', 'away_team', 'team', 'scorer', 'minute', 'own_goal', 'penalty']

shootouts.csv: 675 rows x 5 columns
['date', 'home_team', 'away_team', 'winner', 'first_shooter']

former_names.csv: 36 rows x 4 columns
['current', 'former', 'start_date', 'end_date']


In [13]:
import os
os.environ['WORLD_CUP_REQUIRE_REAL_DATA'] = '1'

# Train the model using the real datasets prepared above.
!python -m src.train_model

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009765 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4625
[LightGBM] [Info] Number of data points in the train set: 39429, number of used features: 27
[LightGBM] [Info] Start training from score -0.715174
[LightGBM] [Info] Start training from score -1.475612
[LightGBM] [Info] Start training from score -1.264947
2026-05-26 12:11:41,714 | INFO | world_cup_simulator | Best model: xgboost
2026-05-26 12:11:41,715 | INFO | world_cup_simulator | Saved model to /content/world-cup-2026-simulator/models/match_prediction_model.joblib


In [ ]:
# Train the model using the real datasets prepared above.
!python -m src.train_model

In [14]:
# Save the trained artifacts to Google Drive.
DRIVE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
for filename in ['match_prediction_model.joblib', 'match_prediction_metadata.json']:
    source_file = MODELS_DIR / filename
    if not source_file.exists():
        raise FileNotFoundError(f'Missing trained artifact: {source_file}')
    shutil.copy2(source_file, DRIVE_MODELS_DIR / filename)
    print(f'Saved {filename} to Google Drive.')

Saved match_prediction_model.joblib to Google Drive.
Saved match_prediction_metadata.json to Google Drive.


In [15]:
# Quick real-team prediction test.
from src.predict import MatchPredictor

predictor = MatchPredictor()
prediction = predictor.predict_match('France', 'Brazil')
print(prediction.probabilities.as_dict())
print('Expected goals:', prediction.expected_home_goals, prediction.expected_away_goals)

{'team_a': 0.3874519053860931, 'draw': 0.3203115963842989, 'team_b': 0.292236498229608}
Expected goals: 1.5112148621114658 1.0677467986524765


## Notes

- This notebook does not use synthetic training data.
- The model is driven by the real match history in `results.csv` and a ranking table derived from that same history.
- The extra Kaggle CSVs are kept in `data/raw/` for future feature work or deeper analysis.
- If you want, I can also update the Streamlit app to surface the new real-data-only training path clearly.